In [1]:
import pyvista as pv
import gstools as gs
import numpy as np
from astropy.io import fits

In [2]:
#pv.set_jupyter_backend("trame")
pv.set_jupyter_backend("client")

In [3]:
N = 256
dim, spacing = (N, N, N), (1,1,1)
grid = pv.ImageData(dimensions=dim, spacing=spacing)

In [4]:
field_synth = 'density'
model_gs    = 'stable'
alpha       = 1.5
len_scale   = [16, 16, 16]
angles      = [0, 0, 0]
seed        = 19970221

In [5]:
model = gs.Stable(dim=3, len_scale=len_scale, angles=angles)
srf = gs.SRF(model, seed=seed)

In [6]:
field = srf.mesh

In [7]:
srf.mesh(grid, points='points', name='random-field')

array([ 0.1176233 ,  0.10621889,  0.73786965, ..., -0.51105276,
        0.63888114,  0.78589739], shape=(16777216,))

In [8]:
#grid.contour(isosurfaces=6).plot(smooth_shading=True, use_transparency=True, opacity=0.3, cmap='coolwarm')

In [9]:
print(type(srf.field))

<class 'numpy.ndarray'>


In [10]:
print(srf.field.shape)

(16777216,)


In [11]:
field_3d = np.asarray(grid.point_data['random-field']).reshape(dim, order='F')

In [12]:
print(type(field_3d))
print(field_3d.shape)
print(field_3d.min())
print(field_3d.max())
print(field_3d.mean())
print(field_3d.std())

<class 'numpy.ndarray'>
(256, 256, 256)
-5.717316344324813
5.655939776435531
-0.028966416902105743
1.0307278585997806


In [13]:
header = fits.Header()

header["MODEL"] = model_gs
header["SEED"]  = seed
header["M3D"]   = alpha

header["LSCALE_X"] = len_scale[0]
header["LSCALE_Y"] = len_scale[1]
header["LSCALE_Z"] = len_scale[2]

header["ANGLE_X"] = angles[0]
header["ANGLE_Y"] = angles[1]
header["ANGLE_Z"] = angles[2]

header["DX"] = spacing[0]
header["DY"] = spacing[1]
header["DZ"] = spacing[2]


In [14]:
name = model_gs + "_3d_" + field_synth 
print(name)

stable_3d_density


In [15]:
fits.writeto(
    name + ".fits",
    field_3d.astype("float32"),
    header=header,
    overwrite=True,
)